# 04 · Kurtosis diagnosis (research IS only)

Other test — **not** a STAR change. Attributes excess kurtosis in STAR S2 book returns
to pair concentration, score-scale extremes, VT leverage, and differing tradable windows.

Per-pair comparisons are reported on **(a) full IS** and **(b) the common overlap window**
so WSO's longer history does not mechanically inflate dominance metrics.

No sealed OOS peek. No ledger / STAR writes.

## 0. Imports & Config

In [ ]:
from __future__ import annotations

import os
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from IPython.display import display

from backtest.s2_coint.diagnosis import enrich_trades, extreme_trades
from backtest.s2_coint.report import load_star_stack, require_star
from backtest.s2_coint.research import (
    DEFAULT_STAR_STACK,
    config_from_stack,
    frozen_pairs_for_universe,
    is_end_for_stack,
    load_s1_weekly,
    load_star_panels,
    load_universe_panels,
    lookbacks_for_bar,
    overlay_kalman_hedge,
    overlay_ols_hedge,
    split_is_oos,
)
from backtest.s2_coint.tearsheet import cvar, fit_mean_abs_score
from strategies.s2_coint.engine import simulate_book
from strategies.s2_coint.metrics import corr_to_s1, metrics_from_returns_inference
from strategies.s2_coint.sizing import pair_scale_from_score

warnings.filterwarnings("ignore", category=FutureWarning)

N_EXTREME = 7
STAR_PATH = DEFAULT_STAR_STACK
stack = load_star_stack(STAR_PATH)
require_star("UNIVERSE_STAR", stack.get("UNIVERSE_STAR"))
require_star("EXIT_STAR", stack.get("EXIT_STAR"))
UNIVERSE = str(stack["UNIVERSE_STAR"])
PAIRS = list(stack.get("PAIRS_STAR") or frozen_pairs_for_universe(UNIVERSE, "1d", root=ROOT))
print("UNIVERSE", UNIVERSE)
print("PAIRS", PAIRS)
print("EXIT_STAR", stack.get("EXIT_STAR"))
print("BREAK_STAR", stack.get("BREAK_STAR"))
print("SIZE_STAR", stack.get("SIZE_STAR"))
print("TREND_STAR", stack.get("TREND_STAR"))
print("VOL_STAR", stack.get("VOL_STAR"))
print("NOTE: STAR stack is read-only in this notebook (other_tests).")

In [ ]:
def _cagr(returns: pd.Series, periods_per_year: float = 252.0) -> float:
    r = pd.to_numeric(returns, errors="coerce").fillna(0.0).astype(float)
    if r.empty:
        return float("nan")
    total = float((1.0 + r).prod())
    years = len(r) / float(periods_per_year)
    if years <= 0 or total <= 0:
        return float("nan")
    return float(total ** (1.0 / years) - 1.0)


def arm_metrics(returns: pd.Series, s1: pd.Series | None = None) -> dict:
    r = pd.to_numeric(returns, errors="coerce").fillna(0.0).astype(float)
    r.index = pd.to_datetime(r.index)
    m = metrics_from_returns_inference(r, periods_per_year=252.0)
    cagr = _cagr(r)
    mdd = float(m.get("max_drawdown", float("nan")))
    calmar = float(cagr / abs(mdd)) if np.isfinite(cagr) and np.isfinite(mdd) and mdd != 0 else float("nan")
    return {
        "ann_sharpe": m.get("ann_sharpe", float("nan")),
        "max_drawdown": mdd,
        "calmar": calmar,
        "cagr": cagr,
        "skew": m.get("skew", float("nan")),
        "excess_kurtosis": m.get("excess_kurtosis", float("nan")),
        "cvar_5pct": cvar(r, alpha=0.05),
        "corr_to_s1": corr_to_s1(r, s1 if s1 is not None else s1_weekly),
        "n_days": m.get("n_days", 0),
    }


def collect_trades(book, panel: pd.DataFrame) -> pd.DataFrame:
    frames = []
    for pid, res in book.pair_results.items():
        if res.trades is None or res.trades.empty:
            continue
        t = res.trades.copy()
        t["pair_id"] = str(pid)
        frames.append(enrich_trades(t, panel, res.returns))
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


def show_extreme(trades: pd.DataFrame, n: int = N_EXTREME, title: str = "") -> None:
    best, worst = extreme_trades(trades, n=n)
    cols = [
        "pair_id", "side_label", "entry_date", "exit_date", "hold_bars",
        "exit_reason", "pnl_pct", "z_entry", "z_exit", "adf_entry", "adf_exit",
    ]
    if title:
        print(title)
    print(f"=== Top {n} trades ===")
    display(best[cols] if not best.empty else best)
    print(f"=== Bottom {n} trades ===")
    display(worst[cols] if not worst.empty else worst)


def metrics_table(rows: dict[str, dict]) -> pd.DataFrame:
    df = pd.DataFrame(rows).T
    order = [
        "ann_sharpe", "max_drawdown", "calmar", "cagr", "skew",
        "excess_kurtosis", "cvar_5pct", "corr_to_s1", "n_days",
    ]
    cols = [c for c in order if c in df.columns] + [c for c in df.columns if c not in order]
    return df[cols]

In [ ]:
bar = str(stack.get("BAR_STAR") or "1d")
lb = lookbacks_for_bar(bar)
# Prefer cached STAR panels when present; else overlay hedge on the research-IS train panel only.
try:
    train_star, _full_star, _manifest = load_star_panels(
        universe=UNIVERSE, bar=bar, pair_ids=PAIRS, root=ROOT
    )
    is_end = is_end_for_stack(stack, train_star)
    is_raw, _oos_unused = split_is_oos(train_star, is_end=is_end)
    del _oos_unused
    panel_src = "cached star train"
except (FileNotFoundError, ValueError) as exc:
    print("star panels unavailable (", type(exc).__name__, ") — using universe train panel")
    train, _full = load_universe_panels(UNIVERSE, bar, PAIRS, root=ROOT)
    is_end = is_end_for_stack(stack, train)
    is_raw, _oos_unused = split_is_oos(train, is_end=is_end)
    del _oos_unused
    panel_src = "universe train"

hedge = str(stack.get("HEDGE_STAR") or "ols")
# Skip re-overlay when star cache already has z / adf columns.
need_overlay = "z" not in is_raw.columns or "adf_pvalue" not in is_raw.columns
if need_overlay:
    if hedge == "kalman":
        is_panel = overlay_kalman_hedge(
            is_raw,
            z_window=lb["z_window"],
            hl_window=lb["hl_window"],
            adf_window=lb["adf_window"],
        )
    else:
        is_panel = overlay_ols_hedge(
            is_raw,
            ols_window=lb["ols_window"],
            z_window=lb["z_window"],
            hl_window=lb["hl_window"],
            adf_window=lb["adf_window"],
        )
else:
    is_panel = is_raw.copy()

is_panel = is_panel.loc[is_panel["pair_id"].astype(str).isin(PAIRS)].copy()
is_panel["date"] = pd.to_datetime(is_panel["date"])

mean_abs = fit_mean_abs_score(is_panel, score_column="z")
s1_weekly = load_s1_weekly(ROOT)
cfg_star = config_from_stack(stack)
print("panel_src", panel_src, "need_overlay", need_overlay)
print("bar", bar, "is_end", is_end)
print("IS rows", len(is_panel), "pairs", is_panel["pair_id"].nunique())
print("frozen IS mean(|z|)", round(mean_abs, 4))
print("cfg", cfg_star)

In [ ]:
book = simulate_book(is_panel, cfg_star, mean_abs_score=mean_abs)
rets = book.returns
base_m = arm_metrics(rets)
print("STAR baseline IS metrics")
display(pd.Series(base_m).to_frame("value"))
show_extreme(collect_trades(book, is_panel), title="STAR baseline extremes")

## 1. Pair Tradable Windows

In [ ]:
windows = []
for pid in PAIRS:
    g = is_panel.loc[is_panel["pair_id"].astype(str) == pid, "date"]
    g = pd.to_datetime(g)
    windows.append({
        "pair_id": pid,
        "first_date": g.min(),
        "last_date": g.max(),
        "n_bars": int(g.nunique()),
        "n_years": float((g.max() - g.min()).days) / 365.25 if len(g) else float("nan"),
    })
win_df = pd.DataFrame(windows)
display(win_df)

overlap_start = max(pd.Timestamp(r["first_date"]) for r in windows)
overlap_end = min(pd.Timestamp(r["last_date"]) for r in windows)
print("common overlap:", overlap_start.date(), "→", overlap_end.date())
if overlap_end < overlap_start:
    raise ValueError("no common overlap across STAR pairs")

def in_overlap(idx) -> pd.Series:
    ix = pd.to_datetime(idx)
    return (ix >= overlap_start) & (ix <= overlap_end)

## 2. Pair PnL Attribution

Full IS vs common overlap. Idle pair-days contribute 0 to the book mean.

In [ ]:
def pair_attribution(book, *, start=None, end=None) -> pd.DataFrame:
    rows = []
    for pid, res in book.pair_results.items():
        r = pd.to_numeric(res.returns, errors="coerce").astype(float)
        r.index = pd.to_datetime(r.index)
        if start is not None:
            r = r.loc[(r.index >= start) & (r.index <= end)]
        r = r.fillna(0.0)
        trades = res.trades
        n_tr = 0 if trades is None or trades.empty else len(trades)
        if trades is not None and not trades.empty and start is not None:
            ed = pd.to_datetime(trades["entry_date"])
            n_tr = int(((ed >= start) & (ed <= end)).sum())
        total = float((1.0 + r).prod() - 1.0) if len(r) else float("nan")
        var = float(r.var(ddof=1)) if len(r) > 1 else float("nan")
        rows.append({
            "pair_id": pid,
            "n_trades": n_tr,
            "n_active_days": int((r.abs() > 0).sum()),
            "total_return": total,
            "ann_vol": float(r.std(ddof=1) * np.sqrt(252)) if len(r) > 1 else float("nan"),
            "variance": var,
            "mean_daily": float(r.mean()) if len(r) else float("nan"),
        })
    out = pd.DataFrame(rows).set_index("pair_id")
    vsum = out["variance"].sum()
    out["var_share"] = out["variance"] / vsum if vsum and np.isfinite(vsum) else float("nan")
    return out.sort_values("total_return", ascending=False)

print("=== Full IS ===")
attr_full = pair_attribution(book)
display(attr_full)
print("=== Common overlap only ===")
attr_ov = pair_attribution(book, start=overlap_start, end=overlap_end)
display(attr_ov)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
attr_full["total_return"].plot(kind="bar", ax=axes[0], color="#1f4e79", title="Full IS total return")
attr_ov["total_return"].plot(kind="bar", ax=axes[1], color="#2e7d32", title="Overlap total return")
for ax in axes:
    ax.axhline(0, color="k", lw=0.8)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Score Scale Distribution

In [ ]:
# Reconstruct entry-time score scales from enriched trades + panel z / adf.
trades_all = collect_trades(book, is_panel)
scale_rows = []
for row in trades_all.itertuples(index=False):
    pid = str(row.pair_id)
    sig = pd.Timestamp(row.signal_date) if pd.notna(row.signal_date) else pd.NaT
    g = is_panel.loc[is_panel["pair_id"].astype(str) == pid].sort_values("date")
    if pd.isna(sig) or g.empty:
        continue
    hit = g.loc[pd.to_datetime(g["date"]) == sig]
    if hit.empty:
        hit = g.loc[pd.to_datetime(g["date"]) <= sig].tail(1)
    if hit.empty:
        continue
    z = float(hit["z"].iloc[0]) if "z" in hit.columns else float("nan")
    adf = float(hit["adf_pvalue"].iloc[0]) if "adf_pvalue" in hit.columns else float("nan")
    scale = pair_scale_from_score(z, adf, size_mode=str(cfg_star.size_mode), mean_abs_score=mean_abs)
    scale_rows.append({
        "pair_id": pid,
        "entry_date": pd.Timestamp(row.entry_date),
        "signal_date": sig,
        "z": z,
        "adf": adf,
        "scale": scale,
        "pnl_pct": float(row.pnl_pct) if pd.notna(row.pnl_pct) else float("nan"),
        "in_overlap": bool(overlap_start <= pd.Timestamp(row.entry_date) <= overlap_end),
    })
scale_df = pd.DataFrame(scale_rows)
print("entry scale summary (full IS)")
display(scale_df["scale"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).to_frame("scale"))
print("per-pair scale")
display(scale_df.groupby("pair_id")["scale"].describe())
print("share of entries with scale > 2 / > 3")
print("gt2", float((scale_df["scale"] > 2).mean()), "gt3", float((scale_df["scale"] > 3).mean()))

fig, ax = plt.subplots(figsize=(8, 3.4))
for pid, g in scale_df.groupby("pair_id"):
    ax.hist(g["scale"].dropna(), bins=25, alpha=0.45, label=pid)
ax.axvline(2, color="C1", ls="--", label="cap 2")
ax.axvline(3, color="C3", ls="--", label="cap 3")
ax.set_title("Entry score scales (IS)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 4. VT Leverage Analysis

In [ ]:
lev = book.leverage
if lev is None or lev.empty:
    print("No leverage series on book (vol_mode may not be s1_vt).")
else:
    lev = lev.astype(float).dropna()
    print("leverage describe")
    display(lev.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).to_frame("leverage"))
    print("days with leverage > 2:", int((lev > 2).sum()), "/", len(lev))
    aligned = pd.concat([
        rets.rename("ret"),
        lev.reindex(rets.index).ffill().rename("leverage"),
    ], axis=1).dropna()
    hi = aligned.loc[aligned["leverage"] > aligned["leverage"].median()]
    lo = aligned.loc[aligned["leverage"] <= aligned["leverage"].median()]
    print("mean |ret| high-lev half", float(hi["ret"].abs().mean()))
    print("mean |ret| low-lev half", float(lo["ret"].abs().mean()))

    fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
    lev.plot(ax=axes[0], title="VT leverage (IS)", color="#1f4e79")
    axes[0].grid(True, alpha=0.3)
    axes[1].hist(lev, bins=30, color="#1f4e79", alpha=0.85)
    axes[1].set_title("Leverage histogram")
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 5. Single-Pair Day Exposure

In [ ]:
# Active = non-zero pair return on that fill date.
active_parts = []
for pid, res in book.pair_results.items():
    r = pd.to_numeric(res.returns, errors="coerce").fillna(0.0)
    r.index = pd.to_datetime(r.index)
    active_parts.append((r.abs() > 1e-12).astype(int).rename(pid))
active = pd.concat(active_parts, axis=1).fillna(0).astype(int)
active["n_active"] = active.sum(axis=1)
active["in_overlap"] = in_overlap(active.index)
active["book_ret"] = rets.reindex(active.index).fillna(0.0)

print("active-pair count (full IS)")
display(active["n_active"].value_counts().sort_index().to_frame("n_days"))
print("active-pair count (overlap)")
display(active.loc[active["in_overlap"], "n_active"].value_counts().sort_index().to_frame("n_days"))

single = active.loc[active["n_active"] == 1]
print("single-pair days outside overlap (likely WSO-only history):",
      int((~single["in_overlap"]).sum()), "/", len(single))

fig, ax = plt.subplots(figsize=(8, 3.4))
for n in sorted(active["n_active"].unique()):
    sub = active.loc[active["n_active"] == n, "book_ret"]
    ax.hist(sub, bins=40, alpha=0.4, label=f"n_active={n}", density=True)
ax.set_title("Book return distribution by # active pairs")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 6. Tail Decomposition

In [ ]:
# Top/bottom daily book returns with pair contributions + leverage.
pair_wide = []
for pid, res in book.pair_results.items():
    r = pd.to_numeric(res.returns, errors="coerce").fillna(0.0)
    r.index = pd.to_datetime(r.index)
    pair_wide.append(r.rename(pid))
wide = pd.concat(pair_wide, axis=1).fillna(0.0)
wide["book"] = rets.reindex(wide.index).fillna(0.0)
if book.leverage is not None and len(book.leverage):
    wide["leverage"] = book.leverage.reindex(wide.index).ffill()
else:
    wide["leverage"] = 1.0

worst_days = wide.nsmallest(N_EXTREME, "book")
best_days = wide.nlargest(N_EXTREME, "book")
print("=== Worst book days ===")
display(worst_days)
print("=== Best book days ===")
display(best_days)

# Dominant pair on each tail day = max |pair contrib|
def dominant_pair(row) -> str:
    cols = [c for c in row.index if c in PAIRS]
    s = row[cols].astype(float).abs()
    return str(s.idxmax()) if len(s) else ""

tail = pd.concat([
    worst_days.assign(tail="worst"),
    best_days.assign(tail="best"),
])
tail = tail.copy()
tail["dominant_pair"] = tail.apply(dominant_pair, axis=1)

# Attach entry score-scale of the trade open on the dominant pair that day.
scale_by_entry = (
    scale_df.assign(
        pair_id=scale_df["pair_id"].astype(str),
        entry_date=pd.to_datetime(scale_df["entry_date"]),
    )
    .drop_duplicates(["pair_id", "entry_date"], keep="last")
    .set_index(["pair_id", "entry_date"])["scale"]
)
trades_for_scale = trades_all.copy()
trades_for_scale["pair_id"] = trades_for_scale["pair_id"].astype(str)
trades_for_scale["entry_date"] = pd.to_datetime(trades_for_scale["entry_date"])
trades_for_scale["exit_date"] = pd.to_datetime(trades_for_scale["exit_date"])


def scale_on_tail_day(day, pid: str) -> float:
    day = pd.Timestamp(day)
    open_tr = trades_for_scale.loc[
        (trades_for_scale["pair_id"] == str(pid))
        & (trades_for_scale["entry_date"] <= day)
        & (trades_for_scale["exit_date"] > day)
    ]
    if open_tr.empty:
        return float("nan")
    key = (str(pid), pd.Timestamp(open_tr.iloc[0]["entry_date"]))
    if key not in scale_by_entry.index:
        return float("nan")
    val = scale_by_entry.loc[key]
    return float(val.iloc[0]) if isinstance(val, pd.Series) else float(val)


tail["score_scale"] = [
    scale_on_tail_day(idx, row.dominant_pair) for idx, row in tail.iterrows()
]
display(
    tail[
        ["book", "leverage", "dominant_pair", "score_scale", "tail"]
        + [c for c in PAIRS if c in tail.columns]
    ]
)

show_extreme(trades_all, title="Trade-level extremes (full IS)")

## 7. Summary

Fill in after running:

1. Does WSO dominance persist on the **common overlap** window, or is it mostly a longer-history artefact?
2. Are HEI / NWS inactive because of ADF entry blocking (few trades) or because trades just don't pay?
3. Do score scales ≫ 2 / VT leverage spikes line up with the worst/best book days?
4. Working hypothesis for kurtosis drivers (concentration / scale / VT / pair dynamics / tradable windows):

In [ ]:
print("Quick facts for the summary cell:")
print("full IS trade counts:", attr_full["n_trades"].to_dict())
print("overlap trade counts:", attr_ov["n_trades"].to_dict())
print("full IS total returns:", attr_full["total_return"].round(4).to_dict())
print("overlap total returns:", attr_ov["total_return"].round(4).to_dict())
print("baseline excess_kurtosis", round(base_m["excess_kurtosis"], 3))
print("baseline skew", round(base_m["skew"], 3))